# Lab 03 — Search by Meaning, Not Just Words

A pretrained model may create embeddings. Your job is to implement cosine similarity and deterministic top-*k* retrieval in NumPy.

**Rules:** CPU only; no vector database; no hosted embedding API; no library nearest-neighbour/search implementation.

In [ ]:
%pip -q install sentence-transformers numpy

In [ ]:
from time import perf_counter
import numpy as np
from sentence_transformers import SentenceTransformer

np.set_printoptions(precision=4, suppress=True)
RNG = np.random.default_rng(42)
MODEL_NAME = "all-MiniLM-L6-v2"

## Challenge corpus

The data is deterministic and deliberately contains paraphrases. It is lab-only data, not the InstaKill domain model.

In [ ]:
TOPICS = {
    "food": [
        "A budget-friendly restaurant opened near the station.",
        "Looking for an affordable place to eat dinner tonight.",
        "The cheapest lunch menu I found this week costs under ten euros.",
    ],
    "running": [
        "Easy jogging route beside the river for beginners.",
        "My first 5K training plan starts with short runs and walks.",
        "Anyone know a calm place for an evening run?",
    ],
    "python": [
        "I finally understood why a Python dictionary needs hashable keys.",
        "Small NumPy experiment: vectorized code was much faster than my loop.",
        "Looking for a clear explanation of Python type hints.",
    ],
    "music": [
        "Quiet jazz playlist for a long study session.",
        "Any recommendations for live music this weekend?",
        "This new album is perfect background sound while reading.",
    ],
    "travel": [
        "Best low-cost train trip for a one-day escape?",
        "I need ideas for a weekend hike without a car.",
        "What should I pack for a short mountain trip in autumn?",
    ],
    "books": [
        "Recommend a short science-fiction novel with a strong ending.",
        "I want a book that explains machine learning without too much math.",
        "Which library has a good quiet corner for reading?",
    ],
}

def make_demo_posts(repetitions=200):
    posts = []
    post_id = 1
    for topic, templates in TOPICS.items():
        for repeat in range(repetitions):
            template = templates[repeat % len(templates)]
            posts.append({
                "post_id": post_id,
                "user_id": (repeat % 37) + 1,
                "topic": topic,
                "content": f"{template} [community note {repeat + 1}]",
            })
            post_id += 1
    RNG.shuffle(posts)
    return posts

posts = make_demo_posts()
texts = [post["content"] for post in posts]
assert len(posts) >= 1_000
print(f"{len(posts)} posts; example: {posts[0]}")

## Baseline: keyword overlap

This baseline is intentionally simple. You may inspect it, but do not use it as your semantic solution.

In [ ]:
def keyword_search(query, posts, k=5):
    query_terms = set(query.lower().split())
    scored = []
    for post in posts:
        score = len(query_terms & set(post["content"].lower().split()))
        scored.append((score, post["post_id"], post))
    scored.sort(key=lambda item: (-item[0], item[1]))
    return [post for score, _, post in scored[:k] if score > 0]

keyword_search("cheap dinner", posts, k=3)

## TODO 1 — Encode text

Load the supplied pretrained encoder and create a normalized embedding matrix for all corpus texts. Keep the model call at this boundary; your retrieval functions below must not load a model or perform I/O.

In [ ]:
# Load the pretrained encoder only at the model boundary and normalize once.
model = SentenceTransformer(MODEL_NAME, device="cpu")
corpus_embeddings = np.asarray(
    model.encode(texts, normalize_embeddings=True, convert_to_numpy=True),
    dtype=np.float32,
)
assert corpus_embeddings.shape[0] == len(posts)


## TODO 2 — Pure retrieval functions

Implement these functions with NumPy. A zero vector must not produce `NaN` or infinity. For equal scores, the lower `post_id` comes first.

In [ ]:
def cosine_similarity(query_vector, corpus_vectors):
    """Return one cosine score for every row in corpus_vectors."""
    query = np.asarray(query_vector, dtype=np.float32)
    corpus = np.asarray(corpus_vectors, dtype=np.float32)
    if query.ndim != 1 or corpus.ndim != 2 or query.shape[0] != corpus.shape[1]:
        raise ValueError("query and corpus dimensions do not match")

    query_norm = np.linalg.norm(query)
    corpus_norms = np.linalg.norm(corpus, axis=1)
    denominators = query_norm * corpus_norms
    dot_products = corpus @ query
    return np.divide(
        dot_products,
        denominators,
        out=np.zeros(corpus.shape[0], dtype=np.float32),
        where=denominators != 0,
    )


def stable_top_k(scores, post_ids, k):
    """Return indices ordered by score descending, then post_id ascending."""
    scores = np.asarray(scores)
    post_ids = np.asarray(post_ids)
    if scores.ndim != 1 or post_ids.ndim != 1 or len(scores) != len(post_ids):
        raise ValueError("scores and post_ids must be one-dimensional and aligned")
    if k < 1:
        raise ValueError("k must be at least 1")
    return np.lexsort((post_ids, -scores))[:k]


def semantic_search(query_embedding, posts, corpus_embeddings, k=5):
    """Return up to k dictionaries with post data and a numeric score."""
    if k < 1:
        raise ValueError("k must be at least 1")
    if len(posts) != len(corpus_embeddings):
        raise ValueError("posts and corpus_embeddings must be aligned")

    scores = cosine_similarity(query_embedding, corpus_embeddings)
    post_ids = np.array([post["post_id"] for post in posts])
    indices = stable_top_k(scores, post_ids, k)
    return [
        {**posts[index], "score": float(scores[index])}
        for index in indices
    ]


## Acceptance checks

Run this cell only after implementing the TODOs. These checks are deliberately small; they verify the retrieval contract, not whether a model is magically perfect.

In [ ]:
tiny_query = np.array([1.0, 0.0])
tiny_corpus = np.array([[1.0, 0.0], [0.0, 1.0], [0.5, 0.0], [0.0, 0.0]])
tiny_scores = cosine_similarity(tiny_query, tiny_corpus)
assert np.allclose(tiny_scores, [1.0, 0.0, 1.0, 0.0])
assert np.isfinite(cosine_similarity(np.array([0.0, 0.0]), tiny_corpus)).all()

tie_indices = stable_top_k(np.array([0.7, 0.9, 0.9]), np.array([7, 5, 2]), k=2)
assert list(tie_indices) == [2, 1], "ties must use ascending post_id"

assert corpus_embeddings.shape[0] == len(posts)
assert np.issubdtype(corpus_embeddings.dtype, np.floating)

query_embedding = model.encode(["cheap dinner near the station"], normalize_embeddings=True)[0]
results = semantic_search(query_embedding, posts, corpus_embeddings, k=5)
assert 1 <= len(results) <= 5
assert all({"post_id", "user_id", "content", "score"} <= set(result) for result in results)
assert all(results[i]["score"] >= results[i + 1]["score"] for i in range(len(results) - 1))
assert all(np.isfinite(result["score"]) for result in results)
print("Acceptance checks passed.")

## Compare search behavior

Use at least three queries. For each, compare the keyword baseline and your semantic search. Record one useful result and one surprising or weak result.

In [ ]:
comparison = []
for query in queries:
    query_embedding = model.encode(
        [query], normalize_embeddings=True, convert_to_numpy=True
    )[0]
    semantic_results = semantic_search(
        query_embedding, posts, corpus_embeddings, k=3
    )
    comparison.append({
        "query": query,
        "keyword": keyword_search(query, posts, k=3),
        "semantic": semantic_results,
    })

for item in comparison:
    print(f"\nQUERY: {item['query']}")
    print("keyword:", [post["content"] for post in item["keyword"]])
    print("semantic:", [
        (result["post_id"], round(result["score"], 4), result["content"])
        for result in item["semantic"]
    ])


RESULTS
baseline: Keyword overlap is fast and transparent, but it misses paraphrases such as "cheap dinner" versus "budget-friendly restaurant" and can return no result when vocabulary differs.
solution: The normalized all-MiniLM-L6-v2 embeddings plus NumPy cosine similarity retrieved semantically related posts for "cheap dinner", "beginner running route", and "learn Python types". The first query demonstrates a vocabulary-changing match: "budget-friendly restaurant" is relevant even without the exact words "cheap dinner". A weak result can occur when a short query is ambiguous, such as "types", because the encoder has limited context.
runtime: CPU-only; corpus encoding is the expensive one-time step, while each query performs a matrix-vector product and deterministic NumPy sort. Measured values are printed by the notebook run.
memory / model size: 1,200 float32 corpus vectors (384 dimensions each) plus the compact all-MiniLM-L6-v2 model; exact process/model memory depends on the runtime.
3 failure cases: ambiguous short queries; negation or sarcasm; rare names and facts not represented by the pretrained model.
what I would try next: benchmark batching and cached norms, evaluate more labeled paraphrase queries, and refresh the in-memory index when new posts are created.
